In [ ]:
import torch
import torch.nn as nn
import os
import json
import sys
import random
import math
from IPython.display import display
# from transformers import AutoTokenizer, AutoModel, ModernBertConfig, ModernBertModel

sys.path.append('/Users/orenm/Desktop/code_projects/BlenderShaderProject/project_files/')

In [ ]:
from Logic.blender_tree_manager import BlenderTreeManager
from Logic.NN_models.data_loaders import ImageCodeDataset
from Logic.NN_models.images_to_code_model import ImageToCodeDecoder
from Logic.bpy_connector import generate_image

In [ ]:
path = '/Users/orenm/BlenderShaderProject/data/'
images_path = os.path.join(path, 'images/')
db_path = os.path.join(path, 'DB/')
temp_img_path = os.path.join(path, 'temp_images/')
active_models_path = os.path.join(path, "active_models/")
mcts_workdir = os.path.join(path, 'mcts_work_dir')
pintrest_images_dir = os.path.join(path, 'pintrest_images_processed_squares')

In [ ]:
IMAGE_EMBEDDER_OUTPUT_DIM = 128
CODES_FILE = '/Users/orenm/BlenderShaderProject/data/datasets/network_managers_strings.json'
ACTIVE_MODELS_PATH = '/Users/orenm/BlenderShaderProject/data/active_models/'
image_embedder_for_texture = 'ep_4_la_7_256_le_0_0001_mo_resnet_fi_128_sc_cosine.pt'
IMAGE_DIR = '/Users/orenm/BlenderShaderProject/data/images/'
CODES_FILE = '/Users/orenm/BlenderShaderProject/data/datasets/network_managers_strings.json'
TOKENIZER_PATH = os.path.join(ACTIVE_MODELS_PATH, "my_tokenizer")
image_embedder_for_texture_path = os.path.join(ACTIVE_MODELS_PATH, image_embedder_for_texture)
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
tokenizer = SpecialTokenTokenizerWrapper(tokenizer)
image_embedder = load_resnet_model(image_embedder_for_texture_path)
MODEL_SAVE_PATH = os.path.join(ACTIVE_MODELS_PATH, "image_to_code_decoder_final.pth")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
strings = json.load(open(CODES_FILE, "r"))
lengths = [len(tokenizer.encode(x)) for x in strings.values()]
MAX_SEQUENCE_LENGTH = max(lengths) + 15

In [ ]:
loaded_model = ImageToCodeDecoder(
    embedder=image_embedder,
    tokenizer=tokenizer,
    vocab_size=tokenizer.vocab_size,
    image_emb_dim=IMAGE_EMBEDDER_OUTPUT_DIM,
    model_dim=768,
    num_layers=16,
    num_heads=12,
    max_seq_len=MAX_SEQUENCE_LENGTH,
    pad_token_id=tokenizer.pad_token_id,
    sos_token_id=tokenizer.sos_token_id,
    eos_token_id=tokenizer.eos_token_id
).to(device)
loaded_model.load_state_dict_from_file(MODEL_SAVE_PATH, device=device)
loaded_model.eval()  # Set to evaluation mode for inference

In [ ]:
dataset = ImageCodeDataset(
    image_dir=IMAGE_DIR,
    codes_filepath=CODES_FILE,
    tokenizer=tokenizer,
    max_seq_len=MAX_SEQUENCE_LENGTH  # Pass MAX_SEQUENCE_LENGTH to the dataset
)

In [ ]:
images_to_test_on = ['21633', '125953', '76803', '98059', '44811', '175119', '169872', '65551', '148626', '170387', '125715', '109588',
                     '101529', '49945', '124446', '154398', '150818', '53413', '18469', '15015', '64681', '145706', '116395', '162091',
                     '107565', '141998', '76079', '71216', '58033', '27700', '31481', '59318', '59702', '52793', '164922', '47803', '170938',
                     '15293', '59074', '143171', '126276', '46277', '39750', '170567', '167369', '19663', '51792', '16595', '64341', '172503',
                     '107608', '163033', '64602', '129242', '21723', '39901', '152413', '151265', '158183', '91115', '57196', '51565', '45295',
                     '115825', '154737', '46580', '34805', '155767', '49784', '159993', '49276', '151806', '57041', '43225', '159596', '12101',
                    '89387', '45401', '38743', '25682', '128718', '132416', '37178', '110873', '92773', '101139', '60374', '35174', '9395',
                    '144665', '9295', '45952']

In [ ]:
for i in images_to_test_on:
    rand_img = os.path.join(IMAGE_DIR, random.choice(images_to_test_on)) + '.png'
    display(Image.open(rand_img))

In [ ]:
rand_img = os.path.join(IMAGE_DIR, random.choice(images_to_test_on)) + '.png'
Image.open(rand_img)

In [ ]:
for img_name in os.listdir(pintrest_images_dir):
    img_path = os.path.join(pintrest_images_dir, img_name)
    display(Image.open(img_path))
    image_tensor = dataset.open_image(img_path).to(device)
    try:
        generated_codes = loaded_model.generate(image_tensor.unsqueeze(1), max_new_tokens=MAX_SEQUENCE_LENGTH, greedy=True)
        code = generated_codes[0]
        nm = BlenderTreeManager.from_str(code)
        generate_image(nm, '/tmp/tmp.png')
        display(Image.open('/tmp/tmp.png'))
    except:
        print('error')